# 02 — Reprodução dos candidatos pelo Pixel Purity Index

Este caderno implementa integralmente o Pixel Purity Index (PPI) e o Spectral Angle Mapper (SAM), sem recorrer ao pacote Python do repositório. O processamento é feito separadamente para cada combinação `classe TerraClass × composto de 16 dias`.

A saída esperada é um conjunto de até quatro pixels espectralmente extremos por data. Esses pixels são **candidatos** a endmember; o PPI não atribui significado biofísico e não decide quais assinaturas representam vegetação fotossinteticamente ativa.

## Preparação

O caderno lê diretamente a configuração YAML e os Parquets WTSS. Todos os caminhos são relativos à raiz detectada automaticamente.

In [ ]:
from pathlib import Path
import json
import hashlib

import numpy as np
import pandas as pd
import yaml


def localizar_repositorio(inicio=None):
    """Localiza a raiz sem pressupor o sistema operacional ou caminho local."""
    atual = Path(inicio or Path.cwd()).resolve()
    for candidato in (atual, *atual.parents):
        if (candidato / "config" / "study.yaml").is_file() and (candidato / "data").is_dir():
            return candidato
    raise FileNotFoundError("A raiz do repositório não foi encontrada.")


RAIZ = localizar_repositorio()
with (RAIZ / "config" / "study.yaml").open(encoding="utf-8") as arquivo:
    CONFIG = yaml.safe_load(arquivo)

CLASSES = {str(codigo).zfill(2): nome for codigo, nome in CONFIG["classes"].items()}
BANDAS = list(CONFIG["bands"])
SAIDA = RAIZ / "outputs" / "standalone"
SAIDA.mkdir(parents=True, exist_ok=True)

print(f"Repositório: {RAIZ}")
print(f"Classes: {', '.join(CLASSES)}")
print(f"Bandas: {', '.join(BANDAS)}")

## Máscara de qualidade

O mesmo indicador \(Q_{i,t}\) auditado no caderno anterior é aplicado antes de cada PPI. As reflectâncias válidas são ordenadas por `systematic_order`, preservando o delineamento congelado. No máximo 1.000 pixels entram em cada data.

In [ ]:
def observacoes_validas(tabela, config):
    clearob = pd.to_numeric(tabela["CLEAROB"], errors="coerce")
    scl = pd.to_numeric(tabela["SCL"], errors="coerce")
    reflectancias = tabela[config["bands"]].apply(pd.to_numeric, errors="coerce").to_numpy(float)
    return clearob.gt(0) & ~scl.isin(config["scl_invalid"]) & np.isfinite(reflectancias).all(axis=1)

## Ângulo espectral

O SAM mede o ângulo entre dois vetores espectrais:

\[
\operatorname{SAM}(x,y)=\cos^{-1}\left(
\frac{x^\mathsf{T}y}{\lVert x\rVert\lVert y\rVert}
\right)\frac{180}{\pi}.
\]

A direção, e não a distância euclidiana, determina a similaridade. Ângulos menores indicam formas espectrais mais parecidas. O protocolo mantém um candidato somente se seu ângulo em relação a todos os já escolhidos for pelo menos 2°. O valor de 2° é uma decisão operacional explícita, usada para evitar completar a lista com duplicatas quase idênticas.

In [ ]:
def angulo_espectral_graus(a, b):
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    denominador = float(np.linalg.norm(a) * np.linalg.norm(b))
    if denominador <= 0:
        return float("nan")
    cosseno = float(np.clip(np.dot(a, b) / denominador, -1.0, 1.0))
    return float(np.degrees(np.arccos(cosseno)))


def selecionar_candidatos_distintos(pixels, escores, quantidade, ordem_mestra, sam_minimo):
    pixels = np.asarray(pixels, dtype=np.float64)
    escores = np.asarray(escores)
    ordem_mestra = np.asarray(ordem_mestra, dtype=np.int64)

    pool = np.flatnonzero(escores > 0)
    pool = pool[np.lexsort((ordem_mestra[pool], -escores[pool]))]
    selecionados = []

    for indice in pool:
        candidato = int(indice)
        angulos = [angulo_espectral_graus(pixels[candidato], pixels[outro]) for outro in selecionados]
        if any(np.isfinite(angulo) and angulo < sam_minimo for angulo in angulos):
            continue
        selecionados.append(candidato)
        if len(selecionados) == int(quantidade):
            break

    return np.asarray(selecionados, dtype=np.int64), int(len(pool))

## Pixel Purity Index

Para uma data, cada espectro \(x_i\) é centralizado pela média multivariada, sem divisão pelo desvio-padrão:

\[
z_i=x_i-\bar{x}.
\]

Em cada uma das 10.000 iterações é gerado um vetor gaussiano \(g_k\), normalizado para comprimento unitário,

\[
r_k=\frac{g_k}{\lVert g_k\rVert},\qquad g_k\sim N(0,I),
\]

e todos os pixels são projetados nessa direção:

\[
p_{ik}=z_i^\mathsf{T}r_k.
\]

O pixel máximo e o pixel mínimo recebem um voto em cada projeção. Portanto, 10.000 projeções produzem 20.000 votos:

\[
\operatorname{PPI}(i)=\sum_{k=1}^{10.000}\left[
\mathbf{1}\{i=\arg\max_j p_{jk}\}+\mathbf{1}\{i=\arg\min_j p_{jk}\}
\right].
\]

O pool contém todos os pixels com escore positivo. Eles são ordenados por escore decrescente e, em caso de empate, pela ordem sistemática. A seed 13 é reiniciada em cada data, tornando cada resultado determinístico sob a configuração congelada.

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class ResultadoPPI:
    indices: np.ndarray
    espectros: np.ndarray
    escores: np.ndarray
    projecoes: int
    seed: int
    lote_projecoes: int
    tamanho_pool: int
    status: str


def executar_ppi(
    pixels,
    quantidade,
    projecoes,
    seed,
    *,
    lote_projecoes=32,
    sam_duplicata=2.0,
    ordem_mestra=None,
    limite_memoria_mb=6144,
):
    valores = np.asarray(pixels, dtype=np.float64)
    if valores.ndim != 2 or valores.shape[0] < int(quantidade) or valores.shape[1] < 2:
        raise ValueError("O PPI requer uma matriz pixels × bandas com linhas suficientes.")

    finitos = np.all(np.isfinite(valores), axis=1)
    valores = valores[finitos]
    if ordem_mestra is None:
        ordem_mestra = np.arange(1, len(valores) + 1, dtype=np.int64)
    else:
        ordem_mestra = np.asarray(ordem_mestra, dtype=np.int64)[finitos]

    centralizados = valores - np.mean(valores, axis=0, keepdims=True)
    escores = np.zeros(len(valores), dtype=np.int32)
    gerador = np.random.default_rng(int(seed))

    limite_bytes = int(limite_memoria_mb) * 1024 * 1024
    memoria_fixa = valores.nbytes + centralizados.nbytes + escores.nbytes + ordem_mestra.nbytes
    lotes = list(dict.fromkeys([int(lote_projecoes), 16, 8, 4, 1]))
    lote_efetivo = next(
        (
            lote
            for lote in lotes
            if lote <= int(lote_projecoes)
            and memoria_fixa + len(valores) * lote * 8 + lote * valores.shape[1] * 8 <= limite_bytes
        ),
        None,
    )
    if lote_efetivo is None:
        raise MemoryError("O limite de memória não comporta um lote de uma projeção.")

    restantes = int(projecoes)
    while restantes:
        lote = min(lote_efetivo, restantes)
        direcoes = gerador.normal(size=(lote, valores.shape[1]))
        normas = np.linalg.norm(direcoes, axis=1, keepdims=True)
        direcoes = np.divide(direcoes, normas, out=np.zeros_like(direcoes), where=normas > 0)
        projetados = centralizados @ direcoes.T
        np.add.at(escores, np.argmax(projetados, axis=0), 1)
        np.add.at(escores, np.argmin(projetados, axis=0), 1)
        restantes -= lote

    indices, tamanho_pool = selecionar_candidatos_distintos(
        valores,
        escores,
        quantidade,
        ordem_mestra,
        sam_duplicata,
    )
    status = "OK" if len(indices) == int(quantidade) else "FEWER_THAN_FOUR_DISTINCT_CANDIDATES"
    return ResultadoPPI(
        indices=indices,
        espectros=valores[indices],
        escores=escores[indices],
        projecoes=int(projecoes),
        seed=int(seed),
        lote_projecoes=int(lote_efetivo),
        tamanho_pool=tamanho_pool,
        status=status,
    )

## Teste controlado da implementação

Antes de processar os dados, o algoritmo é executado duas vezes sobre a mesma matriz sintética. O teste confirma a reprodutibilidade da seed, a conservação dos 20.000 votos e o limite de quatro candidatos. Ele não valida o significado biofísico dos extremos, que continua dependendo da avaliação espectral.

In [ ]:
gerador_teste = np.random.default_rng(2026)
matriz_teste = gerador_teste.uniform(0.01, 0.60, size=(120, len(BANDAS)))
ordem_teste = np.arange(1, len(matriz_teste) + 1)

teste_a = executar_ppi(matriz_teste, 4, 10_000, 13, ordem_mestra=ordem_teste)
teste_b = executar_ppi(matriz_teste, 4, 10_000, 13, ordem_mestra=ordem_teste)

assert np.array_equal(teste_a.indices, teste_b.indices)
assert np.array_equal(teste_a.escores, teste_b.escores)
assert len(teste_a.indices) <= 4
print("Teste determinístico concluído.")

## Execução completa por classe e data

Datas com menos de quatro pixels válidos são registradas como indisponíveis. Entre quatro e onze pixels válidos, o PPI é executado e a advertência `LOW_VALID_N` é preservada. Se o limiar SAM impedir quatro candidatos distintos, apenas os candidatos encontrados são gravados.

Não há calibração por classe nem repetição entre seeds. Essas são decisões metodológicas explícitas; a reprodutibilidade decorre da configuração única e da seed congelada, enquanto a estabilidade entre seeds não é estimada.

In [ ]:
def processar_classe(codigo, config, diretorio_saida):
    ppi = config["ppi"]
    hash_config = config["provenance"]["production_config_hash"]
    fonte = RAIZ / "data" / "wtss" / f"wtss_class_{codigo}.parquet"
    dados = pd.read_parquet(fonte)
    dados["date"] = pd.to_datetime(dados["date"]).dt.strftime("%Y-%m-%d")

    candidatos = []
    manifesto = []
    for data, grupo in dados.groupby("date", sort=True):
        grupo = grupo.sort_values("systematic_order")
        validos = grupo[observacoes_validas(grupo, config)]
        validos = validos.head(int(ppi["max_sampled_pixels"])).reset_index(drop=True)
        n_validos = len(validos)

        base = {
            "class_code": codigo,
            "class_name": config["classes"][codigo],
            "date": data,
            "total_sampled_pixels": int(len(grupo)),
            "valid_pixels": n_validos,
            "endmembers_max": int(ppi["endmembers_max"]),
            "max_sampled_pixels": int(ppi["max_sampled_pixels"]),
            "ppi_seed": int(ppi["seed"]),
            "ppi_projections": int(ppi["projections"]),
            "candidate_pool_method": ppi["candidate_pool_method"],
            "duplicate_sam_threshold_deg": float(ppi["duplicate_sam_threshold_deg"]),
            "stability_assessed": False,
            "stability_status": "STABILITY_NOT_ASSESSED",
            "config_hash": hash_config,
        }

        if n_validos < int(ppi["endmembers_max"]):
            manifesto.append({**base, "status": "INSUFFICIENT_VALID_PIXELS", "candidate_count": 0})
            continue

        resultado = executar_ppi(
            validos[config["bands"]].to_numpy(float),
            quantidade=int(ppi["endmembers_max"]),
            projecoes=int(ppi["projections"]),
            seed=int(ppi["seed"]),
            lote_projecoes=int(ppi["projection_batch"]),
            sam_duplicata=float(ppi["duplicate_sam_threshold_deg"]),
            ordem_mestra=validos["systematic_order"].to_numpy(np.int64),
            limite_memoria_mb=int(ppi["memory_limit_mb"]),
        )

        qa_status = "LOW_VALID_N" if n_validos < int(ppi["low_valid_n_threshold"]) else "OK"
        avisos = [
            valor
            for valor in [qa_status if qa_status != "OK" else "", resultado.status if resultado.status != "OK" else ""]
            if valor
        ]
        manifesto.append(
            {
                **base,
                "status": qa_status if resultado.status == "OK" else resultado.status,
                "warnings": "|".join(avisos),
                "candidate_count": len(resultado.indices),
                "candidate_pool_size": resultado.tamanho_pool,
                "effective_projection_batch": resultado.lote_projecoes,
            }
        )

        data_compacta = data.replace("-", "")
        for posicao, (indice, escore) in enumerate(zip(resultado.indices, resultado.escores), start=1):
            origem = validos.iloc[int(indice)]
            registro = {
                **base,
                "endmember_label": f"EM{posicao:02d}",
                "candidate_id": f"TCAMZ_C{codigo}_D{data_compacta}_PPI_EM{posicao:02d}",
                "point_id": origem["point_id"],
                "systematic_order": int(origem["systematic_order"]),
                "ppi_score": int(escore),
                "candidate_pool_size": resultado.tamanho_pool,
                "effective_projection_batch": resultado.lote_projecoes,
                "qa_status": qa_status,
                "ppi_status": resultado.status,
            }
            registro.update({banda: float(origem[banda]) for banda in config["bands"]})
            candidatos.append(registro)

    tabela_candidatos = pd.DataFrame(candidatos)
    tabela_manifesto = pd.DataFrame(manifesto)
    diretorio_saida.mkdir(parents=True, exist_ok=True)
    tabela_candidatos.to_csv(diretorio_saida / f"candidates_class_{codigo}.csv", index=False, encoding="utf-8")
    tabela_candidatos.to_parquet(diretorio_saida / f"candidates_class_{codigo}.parquet", index=False)
    tabela_manifesto.to_csv(diretorio_saida / f"ppi_manifest_class_{codigo}.csv", index=False, encoding="utf-8")
    tabela_manifesto.to_parquet(diretorio_saida / f"ppi_manifest_class_{codigo}.parquet", index=False)
    return {
        "class_code": codigo,
        "dates": int(len(tabela_manifesto)),
        "candidates": int(len(tabela_candidatos)),
    }


DIRETORIO_CANDIDATOS = SAIDA / "candidates"
resumos_ppi = [processar_classe(codigo, CONFIG, DIRETORIO_CANDIDATOS) for codigo in CLASSES]
pd.DataFrame(resumos_ppi)

## Comparação exata com a versão publicada

A comparação exige igualdade de identificadores, pixels de origem, escores PPI e valores das dez bandas. Não são usadas tolerâncias numéricas para as reflectâncias, porque a mesma entrada e a mesma sequência pseudoaleatória devem produzir os mesmos candidatos.

In [ ]:
comparacoes = []
for codigo in CLASSES:
    reproduzido = pd.read_parquet(DIRETORIO_CANDIDATOS / f"candidates_class_{codigo}.parquet")
    referencia = pd.read_parquet(RAIZ / "data" / "candidates" / f"candidates_class_{codigo}.parquet")
    colunas = ["candidate_id", "point_id", "ppi_score", *BANDAS]
    reproduzido = reproduzido[colunas].sort_values("candidate_id").reset_index(drop=True)
    referencia = referencia[colunas].sort_values("candidate_id").reset_index(drop=True)

    ids_iguais = reproduzido["candidate_id"].equals(referencia["candidate_id"])
    pontos_iguais = reproduzido["point_id"].equals(referencia["point_id"])
    escores_iguais = np.array_equal(reproduzido["ppi_score"].to_numpy(), referencia["ppi_score"].to_numpy())
    espectros_iguais = np.array_equal(
        reproduzido[BANDAS].to_numpy(float), referencia[BANDAS].to_numpy(float)
    )
    assert ids_iguais and pontos_iguais and escores_iguais and espectros_iguais
    comparacoes.append(
        {
            "class_code": codigo,
            "candidates": len(reproduzido),
            "candidate_ids_identical": ids_iguais,
            "point_ids_identical": pontos_iguais,
            "scores_identical": escores_iguais,
            "spectra_identical": espectros_iguais,
        }
    )

comparacoes = pd.DataFrame(comparacoes)
comparacoes.to_csv(SAIDA / "comparacao_candidatos.csv", index=False, encoding="utf-8")
comparacoes